# Flood LSTM — Leakage-Free Corrected Notebook

**Purpose:** corrected, leakage-free version of the original capstone model.

**Dataset:** `flood_dataset.csv`

The held-out test set is isolated before any data-dependent preprocessing.  
For time-series models, chronological splitting is performed **before** scaling and sliding-window construction.  
Validation data are used for early stopping/model selection; the test set is evaluated only after training is complete.

In [ ]:
# Install only if needed in Colab
!pip -q install pandas numpy matplotlib scikit-learn tensorflow

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
DATA_FILE = "flood_dataset.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Please upload", DATA_FILE)
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(DATA_FILE + " was not found.") from e

assert os.path.exists(DATA_FILE), "Missing file: " + DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
invalid_ts = int(df["timestamp"].isna().sum())
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

FEATURES = ["water_level_cm", "rain_intensity_percent", "water_sensor_percent"]
TARGET = "water_level_cm"
SEQUENCE_LENGTH = 10

print("Rows after timestamp cleaning:", len(df))
print("Invalid timestamps removed:", invalid_ts)
print(df[FEATURES].describe())

## Leakage-free split

The raw chronological records are split first:
- 64% training
- 16% validation
- 20% test

Separate windows are then built inside each partition. This prevents the same raw observation from appearing in both training and held-out test windows.

In [ ]:
n = len(df)
train_end = int(0.64 * n)
val_end = int(0.80 * n)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Raw rows -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

x_scaler.fit(train_df[FEATURES])
y_scaler.fit(train_df[[TARGET]])

def build_sequences(part_df, sequence_length=SEQUENCE_LENGTH):
    X_scaled = x_scaler.transform(part_df[FEATURES])
    y_scaled = y_scaler.transform(part_df[[TARGET]]).reshape(-1)

    X_seq, y_seq = [], []
    for i in range(sequence_length, len(part_df)):
        X_seq.append(X_scaled[i-sequence_length:i])
        y_seq.append(y_scaled[i])

    return np.asarray(X_seq, dtype=np.float32), np.asarray(y_seq, dtype=np.float32)

X_train, y_train = build_sequences(train_df)
X_val,   y_val   = build_sequences(val_df)
X_test,  y_test  = build_sequences(test_df)

print("Window shapes")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

assert len(X_test) > 0

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    LSTM(
        32,
        return_sequences=False,
        unroll=True,
        activation="tanh",
        recurrent_activation="sigmoid"
    ),
    Dense(1)
])

model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred_scaled = model.predict(X_test, verbose=0).reshape(-1, 1)

pred_water = y_scaler.inverse_transform(pred_scaled).reshape(-1)
true_water = y_scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1)

mae = mean_absolute_error(true_water, pred_water)
mse = mean_squared_error(true_water, pred_water)
rmse = np.sqrt(mse)
r2 = r2_score(true_water, pred_water)

print(f"MAE : {mae:.6f}")
print(f"MSE : {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2  : {r2:.6f}")

pd.DataFrame({
    "actual_water_level": true_water,
    "predicted_water_level": pred_water
}).to_csv("flood_leakage_free_test_predictions.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(true_water, label="Actual Water Level")
plt.plot(pred_water, label="Predicted Water Level")
plt.title("Flood LSTM: Leakage-Free Held-Out Test Set")
plt.xlabel("Test sequence")
plt.ylabel("Water level")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Flood LSTM Training")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.show()

In [ ]:
model.save("flood_lstm_leakage_free.keras")

meta = {
    "features": FEATURES,
    "sequence_length": SEQUENCE_LENGTH,
    "x_scaler_min": x_scaler.min_.tolist(),
    "x_scaler_scale": x_scaler.scale_.tolist(),
    "y_scaler_min": y_scaler.min_.tolist(),
    "y_scaler_scale": y_scaler.scale_.tolist(),
    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},
    "seed": SEED
}
with open("flood_lstm_leakage_free_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter._experimental_lower_tensor_list_ops = True
converter.experimental_enable_resource_variables = False
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("flood_lstm_leakage_free.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved Keras model, metadata, predictions, and TFLite model.")